# 🚀 Модуль 5. Урок 5. Корреляция и регрессионный анализ: линейная (OLS) и логистическая регрессия

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_05_statistics/lesson_05_correlation_and_regression_ols_logistic.ipynb)

## 📌 Бизнес-контекст и методология
Корреляция не означает причинно-следственную связь (Correlation != Causation). Регрессионный анализ позволяет изолировать влияние факторов при контроле сопутствующих переменных.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd
from scipy import stats

df_saas = pd.read_csv(saas_csv)

# Линейная регрессия (OLS): Зависимость MRR от количества рабочих мест (active_seats)
res = stats.linregress(df_saas['active_seats'], df_saas['mrr'])
print(f"Коэффициент наклона (Slope / Прирост на 1 место): ${res.slope:.2f}")
print(f"Свободный член (Intercept):                     ${res.intercept:.2f}")
print(f"R-квадрат (R²):                                  {res.rvalue**2:.3f}")
print(f"p-value:                                         {res.pvalue:.4e}")


Коэффициент наклона (Slope / Прирост на 1 место): $0.11
Свободный член (Intercept):                     $77.53
R-квадрат (R²):                                  0.000
p-value:                                         6.6817e-01


### 🎯 Практическое задание 1: Интерпретация коэффициента наклона
Определите, на сколько долларов возрастает ожидаемый MRR при подключении 1 дополнительного рабочего места (`slope` при `active_seats`).

In [3]:
### ВАШ КОД ЗДЕСЬ
seat_coef = res.slope
print(f"Прирост MRR на 1 рабочее место: ${seat_coef:.2f}")

assert seat_coef > 0, "Коэффициент должен быть положительным"
print("✅ Тест успешно пройден: модель линейной регрессии оценена!")


Прирост MRR на 1 рабочее место: $0.11
✅ Тест успешно пройден: модель линейной регрессии оценена!


### 💡 Логистическая регрессия для прогнозирования оттока
Для бинарных исходов (ушел в отток / остался: 1 или 0) используется логистическая регрессия (`Logit`), моделирующая логарифм шансов (log-odds).